# Maya 2 Native: Malayalam evaluation for Mixlumo Voice Studio

Run on **Google Colab** (Runtime > Change runtime type > **T4 GPU**) or **Kaggle** (Accelerator **GPU T4 x2**, **Internet On**).
Then **Run all** (never single cells out of order). When it finishes, download **`maya_eval.zip`** (Colab: folder icon on the left, in `/content`; Kaggle: Output panel) and send it back.

What it measures, with the real model (nothing is simulated):
1. Whether the engine runs on a T4 at all (it was built for BF16 GPUs such as A100/H100/RTX 30-40 series), and which settings are needed.
2. Load time and GPU memory.
3. Malayalam speech with all 10 built-in voices that were trained on Malayalam: speed (real-time factor), time to first audio.
4. Emotion tags (documented: `[laughs] [sighs] [whispers] [sad] [excited]`, plus undocumented ones), each **with and without** the tag, so a listener can compare.
5. Streaming.
6. Voice cloning from a reference clip, with an objective speaker-similarity score. The reference is a **synthetic** voice made by the model itself, so no real person's voice is used. (An optional cell lets you clone **your own** voice, only with your consent.)

Model licence: Apache-2.0 (weights and engine). Mimi codec: CC-BY-4.0. Clone only voices you own or have permission to use.

In [ ]:
# 1. Environment
import os, subprocess, sys, json, time, platform, shutil
OUT = "/kaggle/working/maya_eval" if os.path.isdir("/kaggle/working") else os.path.abspath("maya_eval")
os.makedirs(f"{OUT}/clips", exist_ok=True)
RESULTS = {"started": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "steps": {}}
smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
RESULTS["gpu"] = smi
print("Python", platform.python_version(), "| disk free GB:", shutil.disk_usage("/").free // 2**30)
if not smi:
    raise SystemExit("NO GPU ATTACHED. Colab: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all. "
                     "Kaggle: Session options > Accelerator > GPU T4 x2. If no GPU is offered, the free GPU quota is used up for now.")
print("GPU:", smi)

In [ ]:
# 2. Install the official engine (pinned to the commit Mixlumo reviewed) with its pinned PyTorch 2.8 + CUDA 12.8
MAYA_COMMIT = "c6b3768"
if any(m in sys.modules for m in ("torch", "torchvision", "transformers", "maya3")):
    raise SystemExit("PyTorch is already loaded in this session, so the new versions can't take effect. "
                     "Kaggle: Run > Restart & clear cell outputs. Colab: Runtime > Restart session. Then Run all.")
t = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"maya3[server] @ git+https://github.com/MayaResearch/maya2-native@{MAYA_COMMIT}",
                    # the torchvision built for torch 2.8: Kaggle/Colab ship a newer one, and transformers then fails to
                    # import MimiModel ("Could not import module 'MimiModel'")
                    "torchvision==0.23.0",
                    "--extra-index-url", "https://download.pytorch.org/whl/cu128"], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-4000:])
RESULTS["steps"]["install"] = {"ok": r.returncode == 0, "seconds": round(time.time() - t, 1)}
assert r.returncode == 0, "install failed: see the error above"
# check the imports the engine needs, in a separate process (shows the real error if any)
chk = subprocess.run([sys.executable, "-c", "import torch, torchvision, torchaudio; from transformers import MimiModel; "
                      "print('imports OK: torch', torch.__version__, '| torchvision', torchvision.__version__, "
                      "'| torchaudio', torchaudio.__version__)"], capture_output=True, text=True)
print(chk.stdout, chk.stderr[-3000:])
RESULTS["steps"]["imports"] = {"ok": chk.returncode == 0, "output": (chk.stdout + chk.stderr)[-1500:]}
assert chk.returncode == 0, "the engine's imports fail: see the error above"

In [ ]:
# 3. PyTorch check (a fresh import after the install)
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation (set before torch loads)
import torch
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
info = {"torch": torch.__version__, "cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available(),
        "capability": cap, "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
        "gpu_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if torch.cuda.is_available() else None}
print(info)
RESULTS["torch"] = info
if not str(torch.__version__).startswith("2.8"):
    print("WARNING: the notebook is using a different PyTorch than the one just installed. "
          "Use Run > Restart & clear cell outputs, then Run All again.")

### 4. Find settings that fit this GPU

The official settings needed more than the T4's 14.5 GB in the first test. Each attempt below runs in its **own
process**, so it always starts with an empty GPU (a failed attempt can't leave memory behind for the next one).
Attempts go from official to most memory-saving. Each records a GPU-memory timeline and, on failure, where it failed.
The first setting that loads **and** produces real audio is then loaded in this notebook.

In [ ]:
import gc, traceback
import numpy as np, soundfile as sf

MODEL = "maya-research/maya2-native"
PROBE = "നമസ്കാരം, ഇതൊരു പരീക്ഷണമാണ്."   # "Hello, this is a test."
ATTEMPTS = [
    ("official",                  dict(max_seats=1)),
    ("no_cuda_graphs",            dict(max_seats=1, use_graphs=False)),
    ("no_graphs_kv1024",          dict(max_seats=1, use_graphs=False, capacity=1024)),
    ("no_graphs_kv512_no_triton", dict(max_seats=1, use_graphs=False, capacity=512, fuse_rope=False, fused_depth=False)),
]

PROBE_SCRIPT = "\n".join([
    "import json, sys, threading, time, traceback",
    "import numpy as np, soundfile as sf, torch",
    "name, kw, out, model, text = sys.argv[1], json.loads(sys.argv[2]), sys.argv[3], sys.argv[4], sys.argv[5]",
    "rec = {'name': name, 'settings': kw, 'timeline': []}",
    "stage = ['start']; t0 = time.time(); done = []",
    "def sample():",
    "    while not done:",
    "        rec['timeline'].append([round(time.time() - t0, 1), stage[0], round(torch.cuda.memory_allocated() / 2**30, 2),",
    "                                round(torch.cuda.max_memory_allocated() / 2**30, 2)])",
    "        time.sleep(0.5)",
    "threading.Thread(target=sample, daemon=True).start()",
    "try:",
    "    from maya3 import Maya3TTS",
    "    stage[0] = 'load'; t = time.time()",
    "    tts = Maya3TTS.from_pretrained(model, **kw)",
    "    rec['load_seconds'] = round(time.time() - t, 1)",
    "    rec['gb_after_load'] = round(torch.cuda.memory_allocated() / 2**30, 2)",
    "    rec['peak_gb_load'] = round(torch.cuda.max_memory_allocated() / 2**30, 2)",
    "    stage[0] = 'generate'; t = time.time()",
    "    w = np.asarray(tts.generate(text, lang='ml', voice='Anirban', normalize=True), np.float32)",
    "    rec['probe_seconds'] = round(time.time() - t, 2); rec['probe_audio_s'] = round(len(w) / 24000, 2)",
    "    rec['peak_gb'] = round(torch.cuda.max_memory_allocated() / 2**30, 2)",
    "    rec['ok'] = bool(len(w) > 24000 * 0.3 and np.isfinite(w).all() and float(np.abs(w).max()) > 0.01)",
    "    sf.write(out + '/clips/00_probe_' + name + '.wav', w, 24000)",
    "    tts.close()",
    "except BaseException as e:",
    "    rec['ok'] = False; rec['failed_during'] = stage[0]",
    "    rec['error'] = (type(e).__name__ + ': ' + str(e))[:1500]; rec['trace'] = traceback.format_exc()[-4000:]",
    "done.append(1); time.sleep(0.6)",
    "json.dump(rec, open(out + '/attempt_' + name + '.json', 'w'), indent=1, default=str)",
])
open(f"{OUT}/probe.py", "w").write(PROBE_SCRIPT)

RESULTS["load_attempts"] = []
USED, USED_KW = None, None
for name, kw in ATTEMPTS:
    t = time.time()
    p = subprocess.run([sys.executable, f"{OUT}/probe.py", name, json.dumps(kw), OUT, MODEL, PROBE],
                       capture_output=True, text=True, timeout=2400)
    path = f"{OUT}/attempt_{name}.json"
    rec = json.load(open(path)) if os.path.exists(path) else {"name": name, "settings": kw, "ok": False,
                                                              "error": "process died (no result file)",
                                                              "stderr_tail": p.stderr[-3000:]}
    rec["wall_seconds"] = round(time.time() - t, 1)
    RESULTS["load_attempts"].append(rec)
    json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)
    if rec.get("ok"):
        print(f"[{name}] OK  load {rec.get('load_seconds')} s | peak GPU {rec.get('peak_gb')} GB | "
              f"{rec.get('probe_audio_s')} s of audio in {rec.get('probe_seconds')} s")
        USED, USED_KW = name, kw
        break
    peak = max((x[3] for x in rec.get("timeline", [])), default=None)
    print(f"[{name}] FAILED during {rec.get('failed_during')}: {rec.get('error', '')[:300]}  | peak seen {peak} GB")
    for line in (rec.get("trace") or rec.get("stderr_tail") or "").strip().splitlines()[-8:]:
        print("    " + line)

RESULTS["used_settings"] = USED
json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)
if USED is None:
    shutil.make_archive(OUT, "zip", OUT)
    raise SystemExit("No setting fits/works on this GPU. Download maya_eval.zip (Kaggle: Output panel; Colab: folder icon) "
                     "and send it: the memory timelines and errors are the result.")

from maya3 import Maya3TTS
tts = Maya3TTS.from_pretrained(MODEL, **USED_KW)
print("Loaded in this notebook with:", USED, "| voices:", sorted(tts.voices)[:30])

In [ ]:
# 5. Helpers: generate one clip, time it, check it, save it
def stats(w):
    w = np.asarray(w, np.float32)
    if len(w) == 0:
        return {"empty": True}
    frame = 2400
    n = len(w) // frame
    rms = np.sqrt((w[: n * frame].reshape(n, frame) ** 2).mean(1)) if n else np.array([0.0])
    return {"audio_s": round(len(w) / 24000, 2), "peak": round(float(np.abs(w).max()), 3),
            "rms_dbfs": round(float(20 * np.log10(np.sqrt(np.mean(w ** 2)) + 1e-9)), 1),
            "clipped_samples": int((np.abs(w) >= 0.999).sum()), "finite": bool(np.isfinite(w).all()),
            "silent_fraction": round(float((rms < 0.003).mean()), 2)}

CLIPS = []
def clip(fname, text, voice=None, **kw):
    rec = {"file": fname, "text": text, "voice": voice, **{k: v for k, v in kw.items() if k not in ("ref_audio",)}}
    try:
        torch.cuda.synchronize(); t = time.time()
        w, info = tts.generate(text, lang="ml", voice=voice, normalize=True, return_info=True, **kw)
        torch.cuda.synchronize(); dt = time.time() - t
        sf.write(f"{OUT}/clips/{fname}.wav", w, 24000)
        rec.update(stats(w)); rec["gen_seconds"] = round(dt, 2)
        rec["rtf"] = round(dt / max(len(w) / 24000, 1e-6), 3)
        rec["ttfa_s"] = info.get("ttfa_s"); rec["stop_reason"] = info.get("reason")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:800]
    CLIPS.append(rec)
    print(fname, {k: rec.get(k) for k in ("audio_s", "gen_seconds", "rtf", "ttfa_s", "error")})
    return rec

def save():
    RESULTS["clips"] = CLIPS
    RESULTS["peak_gb_total"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    json.dump(RESULTS, open(f"{OUT}/results.json", "w"), indent=1, default=str)

### 6. Malayalam with every built-in voice trained on Malayalam

Test sentences were written for this evaluation (not copied from any source). Meaning in English is in the comments.

In [ ]:
ML_VOICES = ["Ahana", "Vaidehi", "Malini", "Charvi", "Jahnavi", "Anirban", "Nakul", "Vasant", "Siddhant", "Satvik"]
S = {
    "greeting":  "നമസ്കാരം, മിക്സ്ലൂമോ വോയ്സ് സ്റ്റുഡിയോയിലേക്ക് സ്വാഗതം.",              # Hello, welcome to Mixlumo Voice Studio.
    "weather":   "ഇന്ന് നല്ല കാലാവസ്ഥയാണ്, നമുക്ക് പുറത്ത് നടക്കാൻ പോകാം.",          # The weather is nice today, let's go for a walk.
    "kerala":    "കേരളം ദൈവത്തിന്റെ സ്വന്തം നാടാണ്.",                                  # Kerala is God's own country.
    "paragraph": ("മഴ പെയ്യുമ്പോൾ ഞാൻ ജനലിനരികിൽ ഇരുന്ന് ചായ കുടിക്കും. പുറത്ത് മരങ്ങൾ കാറ്റിൽ ആടുന്നു. "
                  "ആ ശബ്ദം കേൾക്കാൻ എനിക്ക് വളരെ ഇഷ്ടമാണ്."),                      # When it rains I sit by the window drinking tea...
    "numbers":   "ഈ ടിക്കറ്റിന് 250 രൂപയാണ്, ബസ് രാവിലെ 7:30-ന് പുറപ്പെടും.",          # This ticket is 250 rupees; the bus leaves at 7:30 am.
    "short":     "ശരി.",                                                                 # OK. (very short: identity may drift)
}
for v in ML_VOICES:
    clip(f"10_voice_{v}_weather", S["weather"], voice=v)
for k in ("greeting", "kerala", "paragraph", "numbers", "short"):
    clip(f"11_{k}_Anirban", S[k], voice="Anirban")
    clip(f"11_{k}_Ahana", S[k], voice="Ahana")
save()

### 7. Emotion tags, each with and without the tag

Listen to each pair (`..._tag` vs `..._plain`). Documented tags are "best effort"; undocumented ones may be ignored
or even read aloud. Note in the report what you hear.

In [ ]:
EMO = [  # (tag, documented?, sentence)
    ("sad",      True,  "എനിക്ക് ഇത് വിശ്വസിക്കാൻ കഴിയുന്നില്ല, അവൻ ഇനി തിരിച്ചു വരില്ല."),   # I can't believe it, he won't come back.
    ("excited",  True,  "നമ്മൾ ജയിച്ചു! എനിക്ക് സന്തോഷം അടക്കാൻ കഴിയുന്നില്ല!"),            # We won! I can't contain my joy!
    ("whispers", True,  "ശബ്ദമുണ്ടാക്കരുത്, കുഞ്ഞ് ഉറങ്ങുകയാണ്."),                          # Don't make noise, the baby is sleeping.
    ("sighs",    True,  "ഇന്നത്തെ ദിവസം വളരെ ക്ഷീണിപ്പിക്കുന്നതായിരുന്നു."),                # Today was very tiring.
    ("laughs",   True,  "അത് കേട്ടപ്പോൾ എനിക്ക് ചിരി വന്നു, ശരിക്കും രസകരമായിരുന്നു."),      # Hearing that made me laugh; it was really funny.
    ("angry",    False, "നീ എന്തിനാണ് എന്നോട് കള്ളം പറഞ്ഞത്?"),                             # Why did you lie to me?
    ("crying",   False, "ദയവായി എന്നെ വിട്ടു പോകരുത്."),                                    # Please don't leave me.
    ("happy",    False, "ഇന്ന് എന്റെ ജന്മദിനമാണ്, എല്ലാവർക്കും നന്ദി!"),                   # Today is my birthday, thank you all!
]
for tag, documented, text in EMO:
    for v in ("Anirban", "Ahana"):
        clip(f"20_{tag}_{v}_tag", f"[{tag}] {text}", voice=v)
        clip(f"20_{tag}_{v}_plain", text, voice=v)
RESULTS["emotion_tags"] = [{"tag": t, "documented": d} for t, d, _ in EMO]
save()

In [ ]:
# 8. Streaming: time to first audio chunk and chunk timing
for v in ("Anirban", "Malini"):
    info, chunks, times = {}, [], []
    t = time.time()
    for pcm in tts.stream(S["paragraph"], lang="ml", voice=v, normalize=True, info=info):
        times.append(round(time.time() - t, 3)); chunks.append(pcm)
    w = (np.concatenate(chunks).astype(np.float32) / 32767.0) if chunks else np.zeros(0, np.float32)
    sf.write(f"{OUT}/clips/30_stream_{v}.wav", w, 24000)
    rec = {"voice": v, "chunks": len(chunks), "first_chunk_s": times[0] if times else None,
           "total_s": times[-1] if times else None, "audio_s": round(len(w) / 24000, 2), "engine_info": info}
    RESULTS.setdefault("streaming", []).append(rec)
    print(rec)
save()

### 9. Voice cloning (synthetic reference: no real person's voice)

The reference is a clip of the built-in voice **Malini** made above, with its exact transcript. The clone then says
different sentences. Speaker similarity (cosine of ECAPA speaker embeddings, the model's own speaker encoder) compares
the clone with Malini and with other built-in voices: a working clone should be closest to Malini.

In [ ]:
ref_path, ref_text = f"{OUT}/clips/10_voice_Malini_weather.wav", S["weather"]
ref_w, _ = sf.read(ref_path, dtype="float32")
RESULTS["clone_reference"] = {"voice": "Malini (synthetic)", "seconds": round(len(ref_w) / 24000, 2), "transcript": ref_text}
print("reference length:", round(len(ref_w) / 24000, 2), "s (3-10 s recommended)")

def emb(w):   # the engine's own ECAPA speaker encoder, run on the engine thread like all its GPU work
    w = np.asarray(w, np.float32)
    return tts.engine.call(lambda: tts._embed(w))
voice_emb = {v: emb(sf.read(f"{OUT}/clips/10_voice_{v}_weather.wav", dtype="float32")[0])
             for v in ML_VOICES if os.path.exists(f"{OUT}/clips/10_voice_{v}_weather.wav")}

clone_results = []
for mode in ("icl_spk", "icl", "spk"):
    try:
        tts.register_voice(f"clone_{mode}", ref_audio=ref_path, ref_text=ref_text if mode != "spk" else None,
                           lang="ml", mode=mode)
    except Exception as e:
        clone_results.append({"mode": mode, "error": f"{type(e).__name__}: {e}"[:500]}); print(mode, e); continue
    for k in ("greeting", "kerala", "paragraph"):
        r = clip(f"40_clone_{mode}_{k}", S[k], voice=f"clone_{mode}")
        if "error" in r:
            continue
        e = emb(sf.read(f"{OUT}/clips/40_clone_{mode}_{k}.wav", dtype="float32")[0])
        sims = {v: round(float(np.dot(e, ve)), 3) for v, ve in voice_emb.items()}
        best = max(sims, key=sims.get)
        clone_results.append({"mode": mode, "sentence": k, "similarity_to_Malini": sims.get("Malini"),
                              "closest_voice": best, "all_similarities": sims})
        print(mode, k, "-> similarity to Malini", sims.get("Malini"), "| closest:", best)
RESULTS["cloning"] = clone_results
save()

### 10. Optional: clone **your own** voice (only with your consent)

1. Record 5-10 s of yourself reading one Malayalam sentence clearly, in a quiet room (WAV or FLAC).
2. Add it to this notebook: **Add Input > Upload > New Dataset**, then put its path below.
3. Type the **exact** sentence you read, and set `I_CONSENT = True` (it is your own voice and you agree to clone it).

Skip this cell if you don't want to.

In [ ]:
I_CONSENT = False
MY_REF = ""            # e.g. "/kaggle/input/my-voice/me.wav"
MY_TEXT = ""           # the exact Malayalam sentence you read
if I_CONSENT and MY_REF and MY_TEXT and os.path.exists(MY_REF):
    tts.register_voice("my_voice", ref_audio=MY_REF, ref_text=MY_TEXT, lang="ml")
    for k in ("greeting", "paragraph"):
        clip(f"50_my_voice_{k}", S[k], voice="my_voice")
    mine = emb(np.asarray(__import__("maya3.api", fromlist=["load_audio"]).load_audio(MY_REF), np.float32))
    for k in ("greeting", "paragraph"):
        e = emb(sf.read(f"{OUT}/clips/50_my_voice_{k}.wav", dtype="float32")[0])
        RESULTS.setdefault("own_voice", []).append({"sentence": k, "similarity_to_reference": round(float(np.dot(e, mine)), 3)})
    print(RESULTS.get("own_voice")); save()
else:
    print("Skipped (no consent / no file).")

In [ ]:
# 11. Summary + zip for download
ok = [c for c in CLIPS if "error" not in c]
RESULTS["summary"] = {
    "settings_used": USED, "clips_ok": len(ok), "clips_failed": len(CLIPS) - len(ok),
    "median_rtf": round(float(np.median([c["rtf"] for c in ok])), 3) if ok else None,
    "median_ttfa_s": round(float(np.median([c["ttfa_s"] for c in ok if c.get("ttfa_s") is not None])), 3) if ok else None,
    "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "nvidia_smi_now": subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                                     capture_output=True, text=True).stdout.strip(),
}
RESULTS["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
save()
print(json.dumps(RESULTS["summary"], indent=1))
shutil.make_archive(OUT, "zip", OUT)
print("Download:", OUT + ".zip", round(os.path.getsize(OUT + ".zip") / 2**20, 1), "MB")